# 1. Importing Libraries

In [21]:
import pandas as pd
import mlflow
import mlflow.sklearn

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, accuracy_score

# 2. Loading Feature Store

In [22]:
X_train = pd.read_csv("../data/feature_store/X_train_scaled.csv")
X_val = pd.read_csv("../data/feature_store/X_val_scaled.csv")
y_train = pd.read_csv("../data/feature_store/y_train.csv").values.ravel()
y_val = pd.read_csv("../data/feature_store/y_val.csv").values.ravel()

# 3. Setting MLFlow Experiment

In [23]:
mlflow.set_tracking_uri("file:../mlruns")
mlflow.set_experiment("Malaria_Model_Tracking")

<Experiment: artifact_location=('file:///Users/LamiorkorBoye/Library/CloudStorage/OneDrive-HochschuleLuzern/Kolli '
 "Likhita I.MSCITDS.2501's files - "
 'AI/FINAL_FOLDER/model_tracking/../mlruns/502191582229715147'), creation_time=1774037569819, experiment_id='502191582229715147', last_update_time=1774037569819, lifecycle_stage='active', name='Malaria_Model_Tracking', tags={}, workspace='default'>

In [24]:
experiments = [
    {"name": "LR_baseline", "C": 1.0, "class_weight": None},
    {"name": "LR_balanced", "C": 1.0, "class_weight": "balanced"},
    {"name": "LR_balanced_C_0_1", "C": 0.1, "class_weight": "balanced"},
    {"name": "LR_balanced_C_10", "C": 10.0, "class_weight": "balanced"},
]

results = []

# 4. Training Logistic Regression

In [25]:
for exp in experiments:
    with mlflow.start_run(run_name=exp["name"]):
        model = LogisticRegression(
            max_iter=1000,
            C=exp["C"],
            class_weight=exp["class_weight"],
            solver="liblinear"
        )
        model.fit(X_train, y_train)

        val_probs = model.predict_proba(X_val)[:, 1]
        val_preds = (val_probs >= 0.5).astype(int)

        auc = roc_auc_score(y_val, val_probs)
        f1 = f1_score(y_val, val_preds)
        precision = precision_score(y_val, val_preds, zero_division=0)
        recall = recall_score(y_val, val_preds, zero_division=0)
        accuracy = accuracy_score(y_val, val_preds)

        mlflow.log_param("model", "LogisticRegression")
        mlflow.log_param("C", exp["C"])
        mlflow.log_param("class_weight", str(exp["class_weight"]))
        mlflow.log_param("solver", "liblinear")

        mlflow.log_metric("val_auc", auc)
        mlflow.log_metric("val_f1", f1)
        mlflow.log_metric("val_precision", precision)
        mlflow.log_metric("val_recall", recall)
        mlflow.log_metric("val_accuracy", accuracy)

        results.append({
            "run_name": exp["name"],
            "C": exp["C"],
            "class_weight": exp["class_weight"],
            "val_auc": auc,
            "val_f1": f1,
            "val_precision": precision,
            "val_recall": recall,
            "val_accuracy": accuracy
        })

results_df = pd.DataFrame(results).sort_values(by="val_auc", ascending=False)
results_df

,run_name,C,class_weight,val_auc,val_f1,val_precision,val_recall,val_accuracy
3,LR_balanced_C_10,10.0,balanced,0.994059,0.979592,0.960000,1.000000,0.988095
0,LR_baseline,1.0,None,0.993364,0.972603,0.959459,0.986111,0.984127
1,LR_balanced,1.0,balanced,0.993210,0.979592,0.960000,1.000000,0.988095
2,LR_balanced_C_0_1,0.1,balanced,0.993056,0.979592,0.960000,1.000000,0.988095


In [26]:
best_model = LogisticRegression(
    max_iter=1000,
    C=1.0,
    class_weight="balanced",
    solver="liblinear"
)
best_model.fit(X_train, y_train)

val_probs = best_model.predict_proba(X_val)[:, 1]

threshold_results = []

for threshold in [0.3, 0.4, 0.5, 0.6]:
    preds = (val_probs >= threshold).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "f1": f1_score(y_val, preds),
        "precision": precision_score(y_val, preds, zero_division=0),
        "recall": recall_score(y_val, preds, zero_division=0)
    })

pd.DataFrame(threshold_results)

,threshold,f1,precision,recall
0,0.3,0.972973,0.947368,1.0
1,0.4,0.979592,0.960000,1.0
2,0.5,0.979592,0.960000,1.0
3,0.6,0.979592,0.960000,1.0


# Model Selection

We evaluated multiple Logistic Regression configurations, including baseline and class-balanced models with different regularization strengths.

All balanced models performed similarly, achieving:

Validation AUC ≈ 0.99

F1 score ≈ 0.98

Recall = 1.0

The baseline model performed slightly worse, confirming the importance of handling class imbalance.

We selected the following model for deployment:

Logistic Regression:

C = 1.0

class_weight = "balanced"

solver = "liblinear"

This configuration provides strong performance while remaining simple and interpretable.

# Threshold Selection

We evaluated classification thresholds between 0.3 and 0.6.

Recall remained 1.0 across all thresholds

Precision decreased slightly at lower thresholds

We selected a threshold of 0.5 for deployment as it provides:
- high precision
- perfect recall
- simplicity and interpretability